# Food Security — A County Column That Is a State Value

The USDA Food Environment Atlas is one spreadsheet with **3,143 county rows**, a FIPS column,
and 280 numeric variables. It looks like a county dataset from top to bottom.

It is not one from top to bottom. **49 of those 280 variables — 17.5% — hold a single value per
state, copied into every county row in that state.** Two entire sheets are like this: every one
of the six food-insecurity variables, and every one of the five soda-tax variables. The
assistance sheet is 33 of 53. The access, stores, local-food and socioeconomic sheets are
genuinely county-level throughout.

Nothing in the file distinguishes them. Same FIPS column, same row count, same numeric type,
adjacent sheets in one workbook.

The cost is not cosmetic. Correlating county food insecurity against county low-access
population gives **r = −0.0657, p = 0.000237, n = 3,124** — a highly significant county-level
finding. Computed at the grain the data actually varies at, the same relationship is
**r = −0.1211, p = 0.397, n = 51**. Nothing at all. The published p-value is small because the
sample size is fictional: there are **44** distinct insecurity values in those 3,124 rows, and a
standard error taken at 3,124 is too small by a factor of about **8.4**.

The test that catches it is one `groupby`.

## Learning Objectives

1. **Test the grain of every column before using it**, rather than trusting the row identifier.
2. **Recognise that a file can mix geographies silently**, and that adjacent columns need not
   share one.
3. **Compute an effective sample size** from distinct values rather than row count.
4. **See what a fictional n does to a p-value**, on real data.

## Prerequisites

- Handbook §3 (denominators, universes and partitions).
- Handbook §8.3 (scale, coverage and construct).
- D04 and handbook §1.8 (CDC PLACES) — the adjacent failure, where county values are modelled
  rather than replicated. Both produce county numbers that were not measured at county level;
  the mechanisms differ and so do the defences.

## Data Sources

**USDA Economic Research Service, Food Environment Atlas**, a single `.xls` retrieved from the
ERS data-access page. Thirteen sheets, nine of which carry county rows.

**No API, no version number, no release date in the file.** The payload hash computed on
retrieval is the only vintage identifier available, and it is the client's, not USDA's — the
weakest pinning position of any source in this corpus, weaker than D27's Zillow hash because
there is no publisher checksum to compare against at all (contrast D19, where Dataverse supplies
one).

The file has also moved: two URLs recorded in earlier surveys now return 404, and the live
location was found by reading the data-access page. The client's error message says so rather
than reporting the dataset as gone.

## 1. Setup

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    verify_digest,
    FEA_DATA_SHEETS,
    classify_geographic_grain,
    fetch_food_environment_atlas,
)

#: The measure this notebook is nominally about, and the sheet it lives on.
INSECURITY_MEASURE = "FOODINSEC_15_17"

#: A genuinely county-varying measure from a different sheet, used in section 6
#: to show what a county-level correlation costs when one side is not one.
ACCESS_MEASURE = "PCT_LACCESS_POP15"

pd.set_option("display.width", 112)
print(f"county data sheets : {len(FEA_DATA_SHEETS)}")
print(f"  {', '.join(FEA_DATA_SHEETS)}")
print(f"measure            : {INSECURITY_MEASURE}")

#: The digest of the payload every figure below was computed from, on this
#: record's own hash basis. The publisher serves no prior vintage, so this pin
#: does not restore one -- it guarantees an upstream change cannot pass
#: unnoticed: the notebook raises instead of reporting different numbers under
#: unchanged prose. Handbook §5.2.
PINNED_INSECURITY_SHEET = "178ae080489832605096165c46decd5df1f03480fcdfdcdb63d9a06875537a1e"


county data sheets : 9
  ACCESS, STORES, RESTAURANTS, ASSISTANCE, INSECURITY, TAXES, LOCAL, HEALTH, SOCIOECONOMIC
measure            : FOODINSEC_15_17


## 2. The sheet

In [2]:
"""Retrieve the food insecurity sheet."""
insecurity, prov = verify_digest(
    fetch_food_environment_atlas(sheet="INSECURITY"),
    PINNED_INSECURITY_SHEET, label='Food Environment Atlas, INSECURITY sheet')
print(f"rows     : {len(insecurity):,}")
print(f"states   : {insecurity['State'].nunique()}")
print(f"columns  : {insecurity.columns.tolist()}")
print(f"vintage  : {prov.vintage}")
print(f"payload  : {prov.content_sha256[:16]}...")
print()
print(insecurity.head(4).to_string(index=False))


rows     : 3,143
states   : 51
columns  : ['FIPS', 'State', 'County', 'FOODINSEC_12_14', 'FOODINSEC_15_17', 'CH_FOODINSEC_14_17', 'VLFOODSEC_12_14', 'VLFOODSEC_15_17', 'CH_VLFOODSEC_14_17']
vintage  : undated bulk release; identified by payload hash only
payload  : 178ae08048983260...

 FIPS State  County  FOODINSEC_12_14  FOODINSEC_15_17  CH_FOODINSEC_14_17  VLFOODSEC_12_14  VLFOODSEC_15_17  CH_VLFOODSEC_14_17
 1001    AL Autauga             16.8             16.3                -0.5              7.2              7.1                -0.1
 1003    AL Baldwin             16.8             16.3                -0.5              7.2              7.1                -0.1
 1005    AL Barbour             16.8             16.3                -0.5              7.2              7.1                -0.1
 1007    AL    Bibb             16.8             16.3                -0.5              7.2              7.1                -0.1


Six numeric columns, 3,143 rows, one per county. The first four rows look like
county data.

## 3. Four counties in Alabama

In [3]:
"""Four Alabama counties."""
alabama = insecurity[insecurity["State"] == "AL"].head(4)
print(alabama[["FIPS", "County", INSECURITY_MEASURE, "VLFOODSEC_15_17"]].to_string(index=False))
print()
distinct = insecurity[INSECURITY_MEASURE].nunique()
print(f"distinct values of {INSECURITY_MEASURE} across {len(insecurity):,} counties: {distinct}")
print(f"states                                                         : "
      f"{insecurity['State'].nunique()}")


 FIPS  County  FOODINSEC_15_17  VLFOODSEC_15_17
 1001 Autauga             16.3              7.1
 1003 Baldwin             16.3              7.1
 1005 Barbour             16.3              7.1
 1007    Bibb             16.3              7.1

distinct values of FOODINSEC_15_17 across 3,143 counties: 44
states                                                         : 51


Autauga, Baldwin, Barbour and Bibb are four different counties with four different
populations, economies and grocery landscapes. Their food insecurity rates are **identical** —
and so are their very-low-food-security rates.

Across all **3,143** county rows there are **44** distinct values of `FOODINSEC_15_17`, against
**51** states. The county column is a state column.

## 4. The test

One `groupby` per column: how many states does this column take exactly one value in? If the
answer is *all of them*, the column does not vary at the geography its rows claim.

In [4]:
"""One groupby is the whole test."""
grain = classify_geographic_grain(insecurity)
print(grain[["column", "rows", "groups", "groups_with_one_value",
             "median_distinct_per_group", "is_group_replicated"]].to_string(index=False))
print()
replicated = int(grain["is_group_replicated"].sum())
print(f"columns holding one value per state, replicated into every county row: "
      f"{replicated} of {len(grain)}")


            column  rows  groups  groups_with_one_value  median_distinct_per_group  is_group_replicated
   FOODINSEC_12_14  3143      51                     51                        1.0                 True
   FOODINSEC_15_17  3143      51                     51                        1.0                 True
CH_FOODINSEC_14_17  3143      51                     51                        1.0                 True
   VLFOODSEC_12_14  3143      51                     51                        1.0                 True
   VLFOODSEC_15_17  3143      51                     51                        1.0                 True
CH_VLFOODSEC_14_17  3143      51                     51                        1.0                 True

columns holding one value per state, replicated into every county row: 6 of 6


**Six of six.** Every numeric column on the insecurity sheet is constant within
every state.

The classifier is deliberately strict about what counts. A column constant in *some* states is
not flagged — a small state genuinely reporting one value is ordinary. A column constant in
*every* state, across fifty-one of them, is not a coincidence about the world; it is a statement
about how the file was assembled.

**[rule]** Before using a column at the geography its rows are labelled with, group by the
coarser geography and count distinct values. It costs four lines and it is the only defence
against this entire class of false precision.

## 5. How much of the Atlas is like this?

The insecurity sheet is not the file. Run the same test on all nine county sheets.

In [5]:
"""How much of the Atlas is like this?"""
rows = []
for sheet in FEA_DATA_SHEETS:
    frame, _ = fetch_food_environment_atlas(sheet=sheet)
    sheet_grain = classify_geographic_grain(frame)
    if sheet_grain.empty:
        continue
    rows.append({
        "sheet": sheet,
        "numeric columns": len(sheet_grain),
        "state-replicated": int(sheet_grain["is_group_replicated"].sum()),
    })
audit = pd.DataFrame(rows)
audit["share"] = audit["state-replicated"] / audit["numeric columns"]
audit = audit.sort_values("share", ascending=False)
print(audit.to_string(index=False, formatters={"share": "{:.1%}".format}))
print()
print(f"total numeric columns : {audit['numeric columns'].sum()}")
print(f"state-replicated      : {audit['state-replicated'].sum()} "
      f"({audit['state-replicated'].sum() / audit['numeric columns'].sum():.1%})")

#: The sheets that are neither wholly replicated nor wholly county-level are the
#: ones an analyst cannot reason about at sheet level at all.
print()
print("mixed sheets, column by column:")
for sheet in ("HEALTH", "RESTAURANTS"):
    frame, _ = fetch_food_environment_atlas(sheet=sheet)
    sheet_grain = classify_geographic_grain(frame)
    replicated = sheet_grain.loc[sheet_grain["is_group_replicated"], "column"].tolist()
    genuine = sheet_grain.loc[~sheet_grain["is_group_replicated"], "column"].tolist()
    print(f"  {sheet}")
    print(f"    state-replicated : {', '.join(replicated)}")
    print(f"    county-level     : {', '.join(genuine)}")


        sheet  numeric columns  state-replicated  share
   INSECURITY                6                 6 100.0%
        TAXES                5                 5 100.0%
   ASSISTANCE               53                33  62.3%
       HEALTH               11                 3  27.3%
  RESTAURANTS               16                 2  12.5%
       ACCESS               41                 0   0.0%
       STORES               36                 0   0.0%
        LOCAL               97                 0   0.0%
SOCIOECONOMIC               15                 0   0.0%

total numeric columns : 280
state-replicated      : 49 (17.5%)

mixed sheets, column by column:


  HEALTH
    state-replicated : PCT_OBESE_ADULTS12, PCT_OBESE_ADULTS17, PCT_HSPA17
    county-level     : PCT_DIABETES_ADULTS08, PCT_DIABETES_ADULTS13, RECFAC11, RECFAC16, PCH_RECFAC_11_16, RECFACPTH11, RECFACPTH16, PCH_RECFACPTH_11_16


  RESTAURANTS
    state-replicated : PC_FFRSALES12, PC_FSRSALES12
    county-level     : FFR11, FFR16, PCH_FFR_11_16, FFRPTH11, FFRPTH16, PCH_FFRPTH_11_16, FSR11, FSR16, PCH_FSR_11_16, FSRPTH11, FSRPTH16, PCH_FSRPTH_11_16, PC_FFRSALES07, PC_FSRSALES07


**49 of 280 numeric columns — 17.5% — are state values in county clothing.**

The distribution is what makes it dangerous. `INSECURITY` and `TAXES` are **100%** replicated;
`ASSISTANCE` is 33 of 53; `HEALTH` is 3 of 11; `RESTAURANTS` 2 of 16. And `ACCESS` (41 columns),
`LOCAL` (97), `STORES` (36) and `SOCIOECONOMIC` (15) are **0%** — genuinely county-level
throughout.

And the mixed sheets are worse than either extreme, because the rule changes **inside** one
sheet. On `HEALTH`, `PCT_OBESE_ADULTS12` and `PCT_OBESE_ADULTS17` are state values copied into
every county row, while `PCT_DIABETES_ADULTS08` and `PCT_DIABETES_ADULTS13` — two columns to
their left, same sheet, same FIPS key, same units — vary by county. A county obesity-versus-
diabetes scatter drawn from this sheet has real variation on one axis and 51 repeated values on
the other, and nothing in the file says so.

`RESTAURANTS` does the same thing with a subtler tell: `PC_FFRSALES12` and `PC_FSRSALES12` are
replicated while `PC_FFRSALES07` — the same measure, five years earlier — is county-level. The
grain changed between vintages of one variable.

So an analyst cannot learn the rule once and apply it. The same workbook, the same FIPS column
and the same merge key give county data on one sheet and state data on the next, and a join
across sheets produces a frame in which some columns mean what they say and others do not.

**[rule]** Classify grain per column, not per file and not per source. A dataset's geography is
a property of its columns.

## 6. What it costs

Take the obvious question — do counties with poor food access have more food insecurity? — and
answer it the way the file invites.

In [6]:
"""What a county-level correlation costs when one side is not county-level."""
access, _ = fetch_food_environment_atlas(sheet="ACCESS")
paired = (insecurity[["FIPS", "State", INSECURITY_MEASURE]]
          .merge(access[["FIPS", ACCESS_MEASURE]], on="FIPS")
          .dropna())

county_r, county_p = stats.pearsonr(paired[INSECURITY_MEASURE], paired[ACCESS_MEASURE])
by_state = paired.groupby("State").agg(
    insecurity=(INSECURITY_MEASURE, "first"), access=(ACCESS_MEASURE, "mean")
)
state_r, state_p = stats.pearsonr(by_state["insecurity"], by_state["access"])

print(f"as published, over county rows : r = {county_r:+.4f}  p = {county_p:.3g}  "
      f"n = {len(paired):,}")
print(f"at the grain that varies       : r = {state_r:+.4f}  p = {state_p:.3g}  "
      f"n = {len(by_state)}")
print()
effective = paired[INSECURITY_MEASURE].nunique()
print(f"distinct values on the insecurity side: {effective}")
print(f"a standard error taken at n = {len(paired):,} rather than the {effective} values "
      f"that exist is too small by about sqrt({len(paired):,}/{effective}) = "
      f"{np.sqrt(len(paired) / effective):.1f}x")


as published, over county rows : r = -0.0657  p = 0.000237  n = 3,124
at the grain that varies       : r = -0.1211  p = 0.397  n = 51

distinct values on the insecurity side: 44
a standard error taken at n = 3,124 rather than the 44 values that exist is too small by about sqrt(3,124/44) = 8.4x


**r = −0.0657 with p = 0.000237 over 3,124 counties.** Significant at any
conventional threshold, and a reader would report it.

**At the grain the data varies at: r = −0.1211 with p = 0.397 over 51 states.** Not significant,
and the point estimate is nearly twice as large — so the county version is not even a
conservative reading of the state one.

The mechanism is the effective sample size. There are **44** distinct insecurity values in those
3,124 rows. Every county in Alabama contributes an identical value, so Alabama supplies one
observation and not sixty-seven. A standard error computed at n = 3,124 is too small by roughly
**√(3,124/44) = 8.4×**, and a p-value computed from it is wrong by orders of magnitude.

This is not a subtle inflation. It is the difference between publishing a finding and not having
one.

**[rule]** Effective sample size is bounded by distinct values, not row count. When a variable
has 44 distinct values, no test on it has more than 44 observations however many rows you
merged it onto.

## Limitations

1. **The classifier detects replication, not its cause.** A column constant within every state
   may be a state estimate copied down, a state-level policy variable that is genuinely constant
   within a state, or a modelling artifact. Section 4 establishes the first; the soda-tax
   variables are almost certainly the second, and being correctly constant does not make them
   county measurements.

2. **"State-replicated" is not "wrong".** A state SNAP participation rate is a real number. The
   defect is the row geography it is served at, and the inferences that invites.

3. **Effective sample size is bounded by distinct values, not equal to them.** √(n/k) is an
   approximation for the standard-error inflation, exact only under equal group sizes. It is used
   here as an order of magnitude.

4. **Section 6 is one pair of variables.** Other pairings will give other numbers; the mechanism
   is general and the magnitude is not.

5. **No published vintage.** The file carries no version or date, so the payload hash is the only
   identifier and two different downloads cannot be compared except by hash.

6. **The variable years are embedded in the names** (`FOODINSEC_15_17`) and are not aligned
   across sheets. Nothing here reconciles them, and a cross-sheet analysis must.

7. **Nothing here measures food insecurity.** The notebook is about the file.

## Key Takeaways

1. **49 of 280 numeric columns are state values replicated into 3,143 county rows** — 17.5% of
   the Atlas.

2. **Two whole sheets are 100% replicated** (`INSECURITY`, `TAXES`) and four are 0%. The rule
   cannot be learned once per file.

3. **44 distinct values across 3,124 rows.** Every Alabama county carries the same number.

4. **The test is one `groupby`**: how many groups does this column take one value in?

5. **p = 0.000237 at the fake grain; p = 0.397 at the real one.** Same data, same relationship.

6. **A standard error at n = 3,124 is 8.4× too small** when the variable has 44 distinct values.

7. **Classify grain per column.** Geography is a property of columns, not of files.

8. **This source has the weakest vintage pinning in the corpus** — a client-computed hash on an
   undated bulk file with no publisher checksum.

## Exercises

1. **Find the ones in between.** Section 5 reports fully-replicated columns. Find columns
   constant within *most* states and decide, for three of them, whether that is a data-assembly
   artifact or a real property of the variable.

2. **Redo a published analysis.** Find any study or dashboard using Atlas county food insecurity,
   and state precisely which of its claims survive at the state grain.

3. **Build the guard.** Write a merge wrapper that refuses to join two Atlas sheets unless every
   numeric column in both passes a grain check at the requested geography, and report what it
   rejects.

4. **Bound the approximation.** Limitation 3 says √(n/k) is exact only under equal group sizes.
   Compute the exact design effect for the insecurity variable and report how far off the
   approximation is.

5. **Check the adjacent failure.** D04 uses CDC PLACES, whose county values are modelled rather
   than replicated. Run `classify_geographic_grain` on a PLACES county extract and report what it
   finds — then say why the result does not clear PLACES of the problem D04 documents.

6. **Write the warning.** In two sentences, draft the note that should appear at the top of the
   `INSECURITY` sheet.

## References

- U.S. Department of Agriculture, Economic Research Service. *Food Environment Atlas.*
  https://www.ers.usda.gov/data-products/food-environment-atlas/
- Feeding America. *Map the Meal Gap* — the modelled county food-insecurity estimates the Atlas's
  insecurity variables derive from. https://map.feedingamerica.org/
- Handbook §1.8 (CDC PLACES — the modelled small-area estimate), §3 (denominators, universes and
  partitions), §8 (scale, coverage and construct).
- `src/krl_tutorials_data/sources.py` (`fetch_food_environment_atlas`, `classify_geographic_grain`)
  and `tests/test_food_atlas.py`.